# Coding practice: interpolate and regress a flow

Flow matching connects two distributions by pairing a sample from each, interpolating between them,
and training a network to predict the difference between the endpoints:

$$x_t=(1-t)\,x_0+t\,x_1,\qquad \text{target}=x_1-x_0.$$

No noise is injected. Build the interpolants and targets below, then compare a noise-to-data pairing
against a data-to-data pairing. The course-page checks use changed cases, so use the tests to
understand the rule instead of transcribing one output.

> **Save your own copy first:** File → Save a copy in Drive.

In [ ]:
import math
import random

random.seed(0)


def interpolate(x0, x1, t):
    """Return the flow-matching input x_t for endpoints x0, x1 at time t in [0, 1]."""
    if not 0.0 <= t <= 1.0:
        raise ValueError("t is a fraction of the way from x0 to x1, so it lies in [0, 1].")
    # TODO: return the linear interpolation of x0 and x1 at t.
    raise NotImplementedError("Return (1 - t) * x0 + t * x1.")


def regression_target(x0, x1):
    """Return the quantity the network is trained to predict for this pair."""
    # TODO: return the difference between the endpoints.
    raise NotImplementedError("Return x1 - x0.")

In [ ]:
x0, x1 = 2.0, 10.0
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    print(f"t={t:>4}: x_t={interpolate(x0, x1, t):>6.2f}   target={regression_target(x0, x1):>6.2f}")

# The endpoints are recovered at t=0 and t=1.
assert abs(interpolate(x0, x1, 0.0) - x0) < 1e-12
assert abs(interpolate(x0, x1, 1.0) - x1) < 1e-12

# The input moves with t, but the target does not: it depends only on the pair.
targets = {regression_target(x0, x1) for _ in range(5)}
assert len(targets) == 1

# The midpoint sits halfway between the endpoints.
assert abs(interpolate(x0, x1, 0.5) - 0.5 * (x0 + x1)) < 1e-12
print("\nChecks passed.")

In [ ]:
# Two pairings, same recipe. One starts from noise (this is the diffusion case);
# the other starts from a second data distribution, which diffusion cannot do.
def make_pairs(source, n=6):
    sharp = [random.gauss(8.0, 1.0) for _ in range(n)]          # the target distribution
    if source == "noise":
        start = [random.gauss(0.0, 1.0) for _ in range(n)]
    elif source == "blurry":
        start = [value + random.gauss(0.0, 0.3) for value in sharp]  # a coupled, related sample
    else:
        raise ValueError("source must be 'noise' or 'blurry'")
    return list(zip(start, sharp))


for source in ("noise", "blurry"):
    pairs = make_pairs(source)
    spread = [abs(regression_target(a, b)) for a, b in pairs]
    print(f"{source:>7} start: mean |target| = {sum(spread) / len(spread):.2f}")

## Interpret the results

Answer in your notes:

1. As `t` varies for one fixed pair, `x_t` changes but the target does not. Why does that make this an
   ordinary supervised regression?
2. The blurry-start pairing produces much smaller targets than the noise-start pairing. What does the
   model therefore have to learn in each case?
3. Diffusion is the special case of this recipe in which the starting distribution is noise. Which line
   in `make_pairs` is the only thing that changes between the two cases?
4. Nothing here adds noise to `x_t`. What is already supplying the variety the model trains on?